# Variational Autoencoder (VAE) - MNIST

Estrutura do notebook:

0. **Etapa 0 (Configuração):** Monta o Google Drive em `MyDrive/VAE`, carrega o dataset MNIST e define a arquitetura `MyVAE` e a função de loss.
1. **Etapa 1 (Treinar):** Treina o modelo e salva os melhores pesos em `MyDrive/VAE/melhor_modelo.pth`.
2. **Etapa 2 (Importar):** Carrega o arquivo `melhor_modelo.pth` salvo no Drive (caso não queira treinar novamente).
3. **Etapa 3 (Testar):** Calcula a loss final no conjunto de teste, exibe a reconstrução das imagens e plota o espaço latente 3D.

## Etapa 0: Configuração, Google Drive e Definição do Modelo

In [ ]:
import os
import shutil
from google.colab import drive

drive.mount('/content/drive/')
os.makedirs('/content/drive/MyDrive/VAE', exist_ok=True)
os.chdir('/content/drive/MyDrive/VAE')

import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MODEL_PATH = 'melhor_modelo.pth'
batch_size = 128

def discretize(sample):
    return 1.0 - sample

def preparar_dados(tam_batch=128):
    transform = transforms.Compose([
        transforms.ToTensor(),
        discretize
    ])
    train_dataset = datasets.MNIST(root='data', train=True, transform=transform, download=True)
    test_dataset = datasets.MNIST(root='data', train=False, transform=transform, download=True)

    train_loader = DataLoader(dataset=train_dataset, batch_size=tam_batch, shuffle=True)
    test_loader = DataLoader(dataset=test_dataset, batch_size=tam_batch, shuffle=False)
    return train_loader, test_loader

class MyVAE(nn.Module):
    def __init__(self, x_dim=784, h_dim1=512, h_dim2=256, z_dim=3):
        super(MyVAE, self).__init__()
        self.x_dim = x_dim
        self.z_dim = z_dim

        # Encoder
        self.enc_conv = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(32),
            nn.SiLU(),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.SiLU(),
            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(128),
            nn.SiLU()
        )
        self.fc1 = nn.Linear(128 * 7 * 7, h_dim1)
        self.ln1 = nn.LayerNorm(h_dim1)
        self.fc2 = nn.Linear(h_dim1, h_dim2)
        self.ln2 = nn.LayerNorm(h_dim2)

        self.fcMU = nn.Linear(h_dim2, z_dim)
        self.fcVAR = nn.Linear(h_dim2, z_dim)

        # Projeção auxiliar para separar melhor as classes no espaço latente durante o treino
        self.classifier = nn.Linear(z_dim, 10)

        # Decoder
        self.fc4 = nn.Linear(z_dim, h_dim2)
        self.ln4 = nn.LayerNorm(h_dim2)
        self.fc5 = nn.Linear(h_dim2, h_dim1)
        self.ln5 = nn.LayerNorm(h_dim1)
        self.fc6 = nn.Linear(h_dim1, 128 * 7 * 7)
        self.ln6 = nn.LayerNorm(128 * 7 * 7)

        self.dec_conv = nn.Sequential(
            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.SiLU(),
            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(32),
            nn.SiLU(),
            nn.Conv2d(32, 1, kernel_size=3, stride=1, padding=1),
            nn.Sigmoid()
        )

    def encoder(self, x):
        x_img = x.view(-1, 1, 28, 28)
        h = self.enc_conv(x_img)
        h = h.view(-1, 128 * 7 * 7)
        h = F.silu(self.ln1(self.fc1(h)))
        h = F.silu(self.ln2(self.fc2(h)))
        return self.fcMU(h), self.fcVAR(h)

    def latentSampling(self, mu, log_var):
        std = torch.exp(0.5 * log_var)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decoder(self, z):
        h = F.silu(self.ln4(self.fc4(z)))
        h = F.silu(self.ln5(self.fc5(h)))
        h = F.silu(self.ln6(self.fc6(h)))
        h = h.view(-1, 128, 7, 7)
        recon_img = self.dec_conv(h)
        return recon_img.view(-1, self.x_dim)

    def forward(self, x):
        mu, log_var = self.encoder(x)
        z = self.latentSampling(mu, log_var)
        return self.decoder(z), mu, log_var

def loss_function(recon_x, x, mu, log_var):
    LR = F.mse_loss(recon_x.view(-1, 784), x.view(-1, 784), reduction='sum')
    KD = -0.5 * torch.sum(1 + log_var - mu.pow(2) - log_var.exp())
    return LR + KD

train_loader, test_loader = preparar_dados(batch_size)
print(f"Ambiente pronto. Diretório: {os.getcwd()} | Dispositivo: {device}")

## Etapa 1: Treinar o Modelo VAE

In [ ]:
epocas = 25
learning_rate = 0.001

vae = MyVAE(x_dim=784, h_dim1=512, h_dim2=256, z_dim=3).to(device)
optimizer = optim.AdamW(vae.parameters(), lr=learning_rate, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epocas, eta_min=1e-5)

melhor_test_loss = float('inf')
historico_train_loss = []
historico_test_loss = []

for epoch in range(1, epocas + 1):
    vae.train()
    train_loss = 0.0
    kl_weight = min(1.0, epoch / 5.0)
    sep_weight = 1.2 if epoch <= int(epocas * 0.8) else 0.3

    for data, labels in train_loader:
        data = data.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()

        recon_batch, mu, log_var = vae(data)
        LR = F.mse_loss(recon_batch.view(-1, 784), data.view(-1, 784), reduction='sum')
        KD = -0.5 * torch.sum(1 + log_var - mu.pow(2) - log_var.exp())
        loss_real = LR + KD

        sep_loss = F.cross_entropy(vae.classifier(mu), labels, reduction='sum')
        loss_treino = LR + kl_weight * KD + sep_weight * sep_loss

        loss_treino.backward()
        train_loss += loss_real.item()
        optimizer.step()

    scheduler.step()
    avg_train_loss = train_loss / len(train_loader.dataset)
    historico_train_loss.append(avg_train_loss)

    vae.eval()
    test_loss = 0.0
    with torch.no_grad():
        for data, _ in test_loader:
            data = data.to(device)
            recon, mu, log_var = vae(data)
            test_loss += loss_function(recon, data, mu, log_var).item()

    avg_test_loss = test_loss / len(test_loader.dataset)
    historico_test_loss.append(avg_test_loss)

    marcador = ''
    if avg_test_loss < melhor_test_loss:
        melhor_test_loss = avg_test_loss
        torch.save(vae.state_dict(), MODEL_PATH)
        marcador = ' [Melhor]'

    print(f"Época {epoch:02d}/{epocas:02d} - Train Loss: {avg_train_loss:.4f} | Test Loss: {avg_test_loss:.4f}{marcador}")

vae.load_state_dict(torch.load(MODEL_PATH, map_location=device))
vae.eval()

print(f"\nTreinamento finalizado. Menor Test Loss: {melhor_test_loss:.4f}")
print(f"Modelo salvo em: {os.path.abspath(MODEL_PATH)}")

## Etapa 2: Importar o Melhor Modelo Treinado

In [ ]:
if os.path.exists(MODEL_PATH):
    vae = MyVAE(x_dim=784, h_dim1=512, h_dim2=256, z_dim=3).to(device)
    vae.load_state_dict(torch.load(MODEL_PATH, map_location=device))
    vae.eval()
    print(f"Modelo carregado de: {os.path.abspath(MODEL_PATH)}")
else:
    print(f"Arquivo '{MODEL_PATH}' não encontrado em {os.getcwd()}.")
    print("Execute a Etapa 1 para treinar ou coloque o arquivo no Drive (MyDrive/VAE/melhor_modelo.pth).")

## Etapa 3: Testar o Modelo

In [ ]:
%matplotlib inline

n_images = 15
amostras_grafico_3d = 500

if 'vae' not in globals():
    raise RuntimeError("O modelo ainda não foi carregado. Execute a Etapa 1 (Treinar) ou a Etapa 2 (Importar).")

vae.eval()
total_loss = 0.0
total_mse = 0.0
total_kl = 0.0

lista_z = []
lista_labels = []
primeiras_originais = None
primeiras_reconstruidas = None
primeiros_labels = None

with torch.no_grad():
    for idx, (data, labels) in enumerate(test_loader):
        data = data.to(device)
        recon, mu, log_var = vae(data)

        lr = F.mse_loss(recon.view(-1, 784), data.view(-1, 784), reduction='sum')
        kd = -0.5 * torch.sum(1 + log_var - mu.pow(2) - log_var.exp())
        total_mse += lr.item()
        total_kl += kd.item()
        total_loss += (lr + kd).item()

        if idx == 0:
            primeiras_originais = data[:n_images].detach().cpu()
            primeiras_reconstruidas = recon[:n_images].detach().cpu()
            primeiros_labels = labels[:n_images].detach().cpu()

        z_batch = vae.latentSampling(mu, log_var)
        lista_z.append(z_batch.detach().cpu().numpy())
        lista_labels.append(labels.detach().cpu().numpy())

n_test = len(test_loader.dataset)
media_loss = total_loss / n_test
media_mse = total_mse / n_test
media_kl = total_kl / n_test

print(f"Test set loss (MSE + KL): {media_loss:.4f}")
print(f"Erro de reconstrução (MSE): {media_mse:.4f}")
print(f"Divergência KL: {media_kl:.4f}")

# 1. Visualização das imagens originais vs. reconstruídas
print("\nRótulos das primeiras imagens de teste:", primeiros_labels.tolist())
image_width = 28
fig_img, axes = plt.subplots(nrows=2, ncols=n_images, sharex=True, sharey=True, figsize=(20, 2.8))
fig_img.suptitle('Originais (topo) vs. Reconstruídas (base)', fontsize=13)

for i in range(n_images):
    axes[0, i].imshow(primeiras_originais[i].view((image_width, image_width)), cmap='binary')
    axes[0, i].set_title(f'{primeiros_labels[i].item()}', fontsize=10)
    axes[0, i].axis('off')

    axes[1, i].imshow(primeiras_reconstruidas[i].view((image_width, image_width)), cmap='binary')
    axes[1, i].axis('off')

plt.tight_layout()
plt.show()

# 2. Visualização 3D do Espaço Latente Z
z_all = np.concatenate(lista_z, axis=0)[:amostras_grafico_3d]
labels_all = np.concatenate(lista_labels, axis=0)[:amostras_grafico_3d]
colors = [str(i) for i in labels_all]
print(f"\nEspaço latente plotado: {z_all.shape}")

fig = px.scatter_3d(
    x=z_all[:, 0],
    y=z_all[:, 1],
    z=z_all[:, 2],
    color=colors,
    category_orders={'color': [str(i) for i in range(10)]},
    labels={'x': 'Z1', 'y': 'Z2', 'z': 'Z3', 'color': 'Dígito'},
    title=f'3D Features Z e os Rótulos (Test Loss: {media_loss:.2f})'
)
fig.update_traces(marker=dict(size=4, opacity=0.85))

annotations = []
for i, label in enumerate(colors[:n_images]):
    annotations.append(dict(x=float(z_all[i, 0]), y=float(z_all[i, 1]), z=float(z_all[i, 2]), text=label, showarrow=True))

fig.update_layout(scene=dict(annotations=annotations), height=650)
fig.show()